# 05 PSO Hyperparameter Tuning

This notebook tunes CNN hyperparameters with PySwarms and compares baseline CNN against CNN+PSO.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

from src.cnn_model import build_cnn_regressor, reshape_for_cnn, train_cnn_model
from src.evaluate import evaluate_predictions, save_metrics
from src.pso_optimizer import decode_cnn_hyperparameters, tune_cnn_with_pso

root_dir = Path.cwd()
processed_dir = root_dir / "data" / "processed"
metrics_dir = root_dir / "results" / "metrics"

candidate_files = [
    processed_dir / "desharnais_processed.csv",
    processed_dir / "cocomo81_processed.csv",
    processed_dir / "china_processed.csv",
]

dataset_path = next((p for p in candidate_files if p.exists()), None)
if dataset_path is None:
    raise FileNotFoundError("Run 02_preprocessing.ipynb first to create processed datasets")

df = pd.read_csv(dataset_path)
target_col = df.columns[-1]
X = df.drop(columns=[target_col]).values.astype(np.float32)
y = df[target_col].values.astype(np.float32)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

X_train_cnn = reshape_for_cnn(X_train)
X_test_cnn = reshape_for_cnn(X_test)

In [ ]:
baseline_cnn = build_cnn_regressor(
    input_length=X_train_cnn.shape[1],
    filters=32,
    kernel_size=3,
    dense_units=64,
    learning_rate=1e-3,
 )

baseline_cnn, _ = train_cnn_model(
    baseline_cnn,
    X_train_cnn, y_train,
    X_test_cnn, y_test,
    epochs=30,
    batch_size=32,
    verbose=0,
 )

baseline_preds = baseline_cnn.predict(X_test_cnn).ravel()
baseline_metrics = evaluate_predictions("cnn_baseline", y_test, baseline_preds)
baseline_metrics

In [ ]:
def objective_fn(particles: np.ndarray) -> np.ndarray:
    scores = []
    for particle in particles:
        hp = decode_cnn_hyperparameters(particle)
        model = build_cnn_regressor(
            input_length=X_train_cnn.shape[1],
            filters=hp["filters"],
            kernel_size=hp["kernel_size"],
            dense_units=hp["dense_units"],
            learning_rate=hp["learning_rate"],
        )
        model, history = train_cnn_model(
            model,
            X_train_cnn, y_train,
            X_test_cnn, y_test,
            epochs=15,
            batch_size=32,
            verbose=0,
        )
        val_mae = history["val_mean_absolute_error"][-1]
        scores.append(val_mae)
    return np.array(scores)

lower_bounds = np.array([8, 2, 8, 1e-4], dtype=float)
upper_bounds = np.array([128, 7, 256, 1e-2], dtype=float)

best_cost, best_position = tune_cnn_with_pso(
    objective_fn=objective_fn,
    dimensions=4,
    lower_bounds=lower_bounds,
    upper_bounds=upper_bounds,
    n_particles=8,
    iters=10,
 )

best_hyperparams = decode_cnn_hyperparameters(best_position)
print("Best PSO validation MAE:", best_cost)
print("Best hyperparameters:", best_hyperparams)

In [ ]:
pso_cnn = build_cnn_regressor(
    input_length=X_train_cnn.shape[1],
    filters=best_hyperparams["filters"],
    kernel_size=best_hyperparams["kernel_size"],
    dense_units=best_hyperparams["dense_units"],
    learning_rate=best_hyperparams["learning_rate"],
 )

pso_cnn, _ = train_cnn_model(
    pso_cnn,
    X_train_cnn, y_train,
    X_test_cnn, y_test,
    epochs=30,
    batch_size=32,
    verbose=0,
 )

pso_preds = pso_cnn.predict(X_test_cnn).ravel()
pso_metrics = evaluate_predictions("cnn_pso", y_test, pso_preds)

comparison = pd.concat([baseline_metrics, pso_metrics], ignore_index=True).sort_values("rmse")
comparison

In [ ]:
comparison_path = save_metrics(
    comparison,
    file_name="cnn_vs_pso_metrics.csv",
    output_dir=metrics_dir,
 )
print("Saved comparison metrics to:", comparison_path)